In [ ]:
import pandas as pd
from pathlib import Path
import os
import ast

In [ ]:
# Define in and out destinations

root = "/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch/Code/MitoticStop" + "/"

outputFolder = root + "Example Output" + "/"

In [ ]:
# Define time interval between frames in minutes
frame_interval = 7

In [ ]:
# Define the input dataframe

df = pd.read_csv(outputFolder + 'MainDataFrame_Lineages_Example.csv')

In [ ]:
# V2 error parsing and conversion


def concatenate_error_analysis_files(root_folder = root):

    root_folder = Path(root_folder)
    
    # Recursively find all matching files
    csv_files = list(root_folder.rglob("*_ErrorAnalysis.csv"))
    
    if not csv_files:
        print("No matching files found.")
        return pd.DataFrame()
    
    # Read and concatenate
    df_list = []
    for file in csv_files:
        try:
            df = pd.read_csv(file)
            df_list.append(df)
        except Exception as e:
            print(f"Error reading {file}: {e}")
    
    if df_list:
        df_errors_V2 = pd.concat(df_list, ignore_index = True)
    else:
        df_errors_V2 = pd.DataFrame()
    
    return df_errors_V2

def add_error_type_flags(df_errors_V2):
    error_types = [
       # "normal",
       # "blank",
        "Laggings",
        "Misaligned",
        "Cytokinesis Failure",
        "Massive Missegregation",
        "DNA bridge",
        "Multipolar Division",
        "Other",
        "Mitotic Death",
        "Slippage",
        "MN"
    ]
    
    for error in error_types:
        df_errors_V2[error] = (
            (df_errors_V2["1st Error Type"] == error) |
            (df_errors_V2["2nd Error Type"] == error)
        )
    
    # Create "any_true" column
    df_errors_V2["any_true"] = df_errors_V2[error_types].any(axis = 1)
    
    return df_errors_V2

df_errors_V2 = concatenate_error_analysis_files()
df_errors_V2 = add_error_type_flags(df_errors_V2)

df_errors_V2["Unique_Source_ID"] = df_errors_V2.Dataset.astype(str) + df_errors_V2.Position.astype(str) + df_errors_V2.Source_ID.astype(str)
df_errors_V2.shape

In [ ]:
df_errors_V2 = df_errors_V2[['Source_ID', 'Dataset', 'Position', 'Laggings',
       'Massive Missegregation', 'DNA bridge', 'Misaligned', 'MN',
       'Multipolar Division', 'Cytokinesis Failure', 'Slippage', 'Other',
       'Mitotic Death', 'any_true']]

df_error = df_errors_V2

In [ ]:
# parse downstream endpoints dataframes after analysis (FIJI)

def getdeaths(filepath, position, dataset):
    deaths = pd.read_csv(filepath, usecols = ["Death", "EndPoint_ID", "Track_ID", "Downstream_IDs"])
    deaths["Position"] = position
    deaths["Dataset"] = dataset 
    return deaths

df_deaths_1 = getdeaths(filepath = root + "Example Data/1_TrackMate_downstream_Filled.csv", position = 1, dataset = "Example")
df_deaths_2 = getdeaths(filepath = root + "Example Data/2_TrackMate_downstream_Filled.csv", position = 2, dataset = "Example")

df_deaths_list = [df_deaths_1, df_deaths_2]

df_deaths_proto = pd.concat(df_deaths_list, ignore_index = True)
df_deaths_proto["EndPoint_ID"] = df_deaths_proto.EndPoint_ID.astype(str)
df_deaths_proto["Death"] = df_deaths_proto.Death.astype(str)

In [ ]:
df_deaths = df_deaths_proto[(df_deaths_proto["Death"] == "TRUE")]
df_leaves = df_deaths_proto[(df_deaths_proto["Death"] == "FALSE")]
df_deaths["Death"] = df_deaths["Death"].str.lower().map({"true": True, "false": False})
df_leaves["Leaves"] = df_leaves["Death"].str.lower().map({"true": False, "false": True})

df["EndPoint_ID"] = df["EndPoint_ID"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Collect all endpoint IDs from df (flatten the lists)
all_df_endpoints = set(e for lst in df["EndPoint_ID"] if isinstance(lst, list) for e in lst)

# Collect all death endpoint IDs
all_dead_endpoints = set(df_deaths.loc[df_deaths["Death"] == True, "EndPoint_ID"])

# Collect all Leaves endpoint IDs
all_leaves_endpoints = set(df_leaves.loc[df_leaves["Leaves"] == True, "EndPoint_ID"])

print("Number of endpoints in df:", len(all_df_endpoints))
print("Number of death endpoints:", len(all_dead_endpoints))
print("Number of death overlaps:", len(all_df_endpoints & all_dead_endpoints))
print("Number of leaves endpoints:", len(all_leaves_endpoints))
print("Number of leaves overlaps:", len(all_df_endpoints & all_leaves_endpoints))

In [ ]:
dead_map = (
    df_deaths[df_deaths["Death"]]
    .groupby(["Track_ID", "Dataset"])["EndPoint_ID"]
    .apply(set)
)

leaves_map = (
    df_leaves[df_leaves["Leaves"]]
    .groupby(["Track_ID", "Dataset"])["EndPoint_ID"]
    .apply(set)
)

def count_dead_offspring(row):
    dead_endpoints = dead_map.get((row["Track_ID"], row["Dataset"]), set())
    endpoints = row["EndPoint_ID"]

    if isinstance(endpoints, list):
        return sum(e in dead_endpoints for e in endpoints)
    return 0

df["N_dead_offspring"] = df.apply(count_dead_offspring, axis=1)

def count_leaving_offspring(row):
    leaves_endpoints = leaves_map.get((row["Track_ID"], row["Dataset"]), set())
    endpoints = row["EndPoint_ID"]

    if isinstance(endpoints, list):
        return sum(e in leaves_endpoints for e in endpoints)
    return 0

df["N_leaving_offspring"] = df.apply(count_leaving_offspring, axis=1)

# Final (inner) merge with error dataframe to only include evaluated divisions
df = df.merge(df_error, how = "inner")
df.shape

In [ ]:
def getonsets(filepath, position, dataset):
    onsets = pd.read_csv(filepath)
    onsets["Source_ID"] = onsets.Label.str.split(":").str.get(1).astype(int)
    onsets["Position"] = position
    onsets["Dataset"] = dataset
    onsets.rename(columns = {"Slice": "Frame_onset"}, inplace = True)
    onsets["Frame_onset"] = onsets.Frame_onset - 1 # this corrects for the +1 frame set by FIJI
    return onsets

df_onsets_1 = getonsets(filepath = root + "Example Data/1_TrackMate_onsets.csv", position = 1, dataset = "Example")
df_onsets_2 = getonsets(filepath = root + "Example Data/2_TrackMate_onsets.csv", position = 2, dataset = "Example")

df_onsets_list = [df_onsets_1, df_onsets_2]

df_onsets = pd.concat(df_onsets_list, ignore_index = True)

In [ ]:
df["Frame"] = df.Frame.astype(int) + 1


df = df.merge(df_onsets, how = "inner") 

df.shape

In [ ]:
def getInterphaseDuration(x, dataframe = df):
    # should be the same as cell_cycle - mitotic duration
    daughter_time = x.Frame_onset
    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID) 
        ]
        if mother_row["Frame"].shape[0] == 1:   
            mother_time = mother_row["Frame"].item()
            interphase_duration = (int(daughter_time) - int(mother_time)) * 7
            return interphase_duration
        else:
            return None


def getMitoticDurationMother(x, dataframe = df):
    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID)
        ]
        
        if len(mother_row) == 1:
            return mother_row.iloc[0].Mitotic_duration_mins
        else:
            return None

def getMetaphaseArrest(x):
    if x < 30:
        return "000-030 min"
    elif (x < 60) & (x > 30):
        return "030-060"
    elif (x < 90) & (x > 60):
        return "060-090"
    elif (x < 120) & (x > 90):
        return "090-120"
    else:
        return "> 120 min"

def getMetaphaseArrest_mother(x):
    if x > 0:
        if x < 30:
            return "000-030 min"
        elif (x < 60) & (x > 30):
            return "030-060"
        elif (x < 90) & (x > 60):
            return "060-090"
        elif (x < 120) & (x > 90):
            return "090-120"
        else:
            return "> 120 min"
    else:
        pass

df["Frames_in_mitosis"] = (df.Frame - df.Frame_onset.astype(int))
df["Mitotic_duration_mins"] = df.Frames_in_mitosis * frame_interval

df["Interphase_duration_mins"] = df.apply(getInterphaseDuration, axis = 1) 
df["Interphase_duration_hrs"] = df.Interphase_duration_mins / 60
df["Mitotic_duration_hrs"] = df.Mitotic_duration_mins / 60

df["Mother_Mitotic_duration_mins"] = df.apply(getMitoticDurationMother, axis = 1)
df["Mother_arrested"] = df.Mother_Mitotic_duration_mins.apply(getMetaphaseArrest_mother)
df["Metaphase_arrested"] = df.Mitotic_duration_mins.apply(getMetaphaseArrest)
df["Frame_onset_hrs"] = df.Frame_onset * frame_interval / 60 # Augmin depletion time at NEB
df["Frame_hrs"] = df.Frame * frame_interval / 60 # Augmin depletion time at anaphase

# Create Bins (one bin for every 12 h of depletion, binning cells with the same onset of mitotis)
interval_range_depletion = pd.interval_range(start = 12, freq = 24, end = 84) 
df['Augmin_depletion_bin'] = pd.cut(df['Frame_onset_hrs'], bins = interval_range_depletion).astype(str)

# Create Bins (one bin for every 30 min of mitotic duration) # kind of the same as "Metaphase_arrested"
interval_range_mitosis = pd.interval_range(start = 0, freq = 30, end = 900) 
df['Mitotic_duration_bin'] = pd.cut(df['Mitotic_duration_mins'], bins = interval_range_mitosis).astype(str)

df["Unique_Track_ID"] = df.Dataset.astype(str) + df.Position.astype(str) + df.Track_ID.astype(str)
df["Unique_Source_ID"] = df.Dataset.astype(str) + df.Position.astype(str) + df.Source_ID.astype(str)

# Drop annotations with mistakes
df = df.drop(df[df.Cell_Cycle_hours < 1].index)
df = df.drop(df[df.Mitotic_duration_mins < 8].index)
df = df.drop(df[df.Mother_Mitotic_duration_mins < 8].index)

In [ ]:
# Correct daughter deaths with mitotic deaths

def correct_death(x):
    if x["Mitotic Death"] == True:
        return x.N_dead_offspring - 2
    else:
        return x.N_dead_offspring

def death_bool(x):
    if x > 0:
        return True
    else:
        return False

df["N_dead_offspring"] = df.apply(correct_death, axis = 1) 
df["N_dead_offspring_bool"] = df.N_dead_offspring.apply(death_bool)

In [ ]:
def conditions(x):
    if x == "siHAUS6":
        return "2_siHAUS6"
    if x == "siLUC1siHAUS6":
        return "3_siLUC1siHAUS6"
    if x == "siUSP28siHAUS6":
        return "4_siUSP28siHAUS6"
    if x == "siMock":
        return "0_siMock"
    if x == "F1020FBS":
        return "F10"
    if x == "Monastrol":
        return "Monastrol"
    else:
        return "1_siLUC1"

df["Condition"] = df.Condition.apply(conditions)

In [ ]:
df.to_csv(outputFolder + "MainDataFrame_MitoticStopwatch_Example.csv")

print("Finished.")